# MUMC 베이스라인 재현 — VQA-RAD + SLAKE

논문: *Masked Vision and Language Pre-training with Unimodal and Multimodal Contrastive Losses for Medical VQA*
(arXiv:2307.05314) — 답 후보를 순위 매기는 MedVQA 모델.

**재현 목표 (Expert-CFG 논문 Table 1 기준)**

| 데이터셋 | Open | Closed | Overall |
|---|---|---|---|
| VQA-RAD | 71.50 | 84.20 | 79.20 |
| SLAKE   | — (분리 미기재) | — | **84.90** |

Repo: https://github.com/pengfeiliheu/mumc

## 구성
| 절 | 내용 |
|---|---|
| 0. 환경 준비 | repo clone, 의존성(torch 는 건드리지 않음), 코드 패치 |
| 1. 사전학습 체크포인트 | `med_pretrain_29.pth` |
| 2. 데이터 — VQA-RAD | `data_RAD/{trainset,testset,answer_all_list}.json, images/` |
| 3. 데이터 — SLAKE | `data_Slake/en/slake_{train,val,test}.json, answer_trainval_list.json`, `data_Slake/imgs/` |
| 4. 설정 패치 + 학습 — VQA-RAD | `train_vqa.py --dataset_use rad` |
| 5. 학습 — SLAKE | `train_vqa.py --dataset_use slake` |
| 6. 결과 | epoch 별 예측 JSON 을 Phi 노트북과 같은 지표로 재채점 |

**코드를 읽고 확인한 사항 (실행해 본 것은 아님)**
- `configs/VQA.yaml` 의 모든 경로는 `/mnt/sda/lpf/data/vqa/` 로 시작하고 구조가 `data_RAD/…`, `data_Slake/en/…`, `data_Slake/imgs/` 입니다.
  → 같은 구조로 `/content/mumc_data/` 아래에 만들고 접두어만 한 번에 치환합니다 (YAML 을 다시 쓰지 않음: PyYAML 로 dump 하면 `2e-5` 가 문자열이 되는 문제 방지).
- `train_vqa.py` 는 **`--output_dir` 을 무시하고** `/mnt/sda/lpf/weights/output/V2/vqa/` 로 덮어쓰며, `import ruamel_yaml` 은 pip 환경에 없습니다 → 패치.
- **`--evaluate` 는 평가를 하지 않습니다** (루프 첫 줄에서 `break`). 레포 원본은 매 epoch 마다 test 셋을 평가해 `result/*_vqa_result_<epoch>.json` 으로 저장하지만, 이 노트북은 시간을 줄이려고 **마지막 epoch 에서만 평가**하도록 패치합니다(`EVAL_EVERY=N` 이면 N epoch 마다 추가 평가). 그래서 test 로 epoch 를 고르는 일이 없습니다. 모델 체크포인트는 `--is_save_path` 가 켜져 있고 epoch>20 일 때만 저장되므로 이 노트북은 예측 JSON 으로 채점합니다.
- **val 셋이 없어서** 레포의 "best epoch" 는 test 로 고른 값입니다 → 아래에서 마지막 epoch 점수를 주 결과로, best 는 참고용(낙관적)으로 분리해서 기록합니다.
- 데이터 키: `image_name, qid, question, answer, answer_type, question_type` (`vqa_dataset.py`, `vqaTools`).
- VQA-RAD 의 답 후보 목록은 YAML 이 `answer_all_list.json` (train+test 의 모든 답), SLAKE 는 `answer_trainval_list.json` (train+val) 입니다 — 레포 방식을 따릅니다.
- `requirements.txt` 는 torch 1.10 / transformers 4.8.1 시대 고정이라 Colab 최신 환경에서는 설치되지 않습니다. 이 노트북은 torch 를 그대로 두고 timm 0.4.12 와 옛 transformers 를 설치한 뒤 **import 스모크 테스트**를 통과해야 진행합니다 (통과 여부는 미검증).

## 0. 환경 준비

In [16]:
import os, subprocess
if not os.path.isdir('/content/mumc'):
    subprocess.run(['git', 'clone', 'https://github.com/pengfeiliheu/mumc.git', '/content/mumc'], check=True)
os.chdir('/content/mumc')
print(os.getcwd())

/content/mumc


In [15]:
# torch/torchvision 은 Colab 기본을 그대로 쓰고, 나머지만 설치 (실패는 숨기지 않음)
# transformers 4.30.2 는 tokenizers<0.14 를 요구하는데 Python 3.12 용 wheel 이 없어 소스 빌드에 실패한다 → 4.43.0 (wheel 있음, xbert.py 의 import 는 코드로 확인)
!pip install -q timm==0.4.12 transformers==4.43.0 ruamel.yaml==0.17.21 opencv-python-headless scikit-image scipy gdown

In [17]:
# 코드 패치 (문자열이 없으면 assert 로 즉시 알림)
p = '/content/mumc/train_vqa.py'
s = open(p).read()

old_imp = 'import ruamel_yaml as yaml'
old_out = "args.output_dir = '/mnt/sda/lpf/weights/output/V2/vqa/' + args.dataset_use + args.output_suffix"
if old_imp in s or old_out in s:
    assert old_imp in s and old_out in s, '원본 문자열이 일부만 있습니다 — train_vqa.py 를 확인하세요'
    s = s.replace(old_imp, 'import ruamel.yaml as yaml')
    s = s.replace(old_out, "args.output_dir = os.environ['MUMC_OUT_DIR']")
    open(p, 'w').write(s)
    print('train_vqa.py 패치 완료')
else:
    assert "os.environ['MUMC_OUT_DIR']" in s, '패치 상태를 알 수 없습니다'
    print('이미 패치됨')

# models/vision/mae.py: numpy>=1.24 에서 제거된 np.float 를 float 로 (해당 경로를 안 쓰면 무해)
p2 = '/content/mumc/models/vision/mae.py'
s2 = open(p2).read()
if 'dtype=np.float)' in s2:
    open(p2, 'w').write(s2.replace('dtype=np.float)', 'dtype=float)'))
    print('mae.py np.float 패치 완료')

# models/xbert.py: 새 transformers(4.43)에서 add_code_sample_docstrings 의 인자 이름이 바뀌어(tokenizer_class 제거) import 시 TypeError 가 난다.
# 이 두 데코레이터는 문서 문자열만 만드므로 아무 동작도 하지 않는 함수로 대체한다 (모델 동작과 무관).
p3 = '/content/mumc/models/xbert.py'
s3 = open(p3).read()
_MARK = '# [patch] docstring decorators'
if _MARK not in s3:
    _anchor = 'from transformers.models.bert.configuration_bert import BertConfig\n'
    assert _anchor in s3, 'xbert.py 의 import 줄을 찾지 못했습니다'
    s3 = s3.replace(_anchor, _anchor + _MARK + '\n'
                    'def add_code_sample_docstrings(*a, **k):\n    return lambda fn: fn\n\n'
                    'def replace_return_docstrings(*a, **k):\n    return lambda fn: fn\n\n', 1)
    open(p3, 'w').write(s3)
    print('xbert.py 패치 완료')
else:
    print('xbert.py 이미 패치됨')

# models/tokenization_bert.py: 새 transformers 의 PreTrainedTokenizer.__init__ 은 안에서 get_vocab() 을 부르는데,
# 이 파일은 super().__init__() 뒤에 self.vocab 을 만들어서 AttributeError('vocab') 가 난다.
# → vocab 과 토크나이저 구성요소를 super().__init__() 앞으로 옮긴다 (self.unk_token 은 아직 없으므로 인자 unk_token 사용).
p4 = '/content/mumc/models/tokenization_bert.py'
s4 = open(p4).read()
_MARK4 = '# [patch] vocab before super'
if _MARK4 not in s4:
    _sup = '        super().__init__(\n            do_lower_case=do_lower_case,'
    _a = s4.index('        if not os.path.isfile(vocab_file):')
    _end_txt = 'self.wordpiece_tokenizer = WordpieceTokenizer(vocab=self.vocab, unk_token=self.unk_token)\n'
    _b = s4.index(_end_txt) + len(_end_txt)
    _block = s4[_a:_b]
    assert _sup in s4 and s4.index(_sup) < _a, 'tokenization_bert.py 구조가 예상과 다릅니다'
    _block_new = _MARK4 + '\n' + _block.replace('unk_token=self.unk_token', 'unk_token=unk_token')
    s4 = s4[:_a] + s4[_b:]
    s4 = s4.replace(_sup, _block_new + _sup, 1)
    open(p4, 'w').write(s4)
    print('tokenization_bert.py 패치 완료')
else:
    print('tokenization_bert.py 이미 패치됨')

# train_vqa.py: 매 epoch 마다 test 를 평가하는 것을 마지막 epoch 에서만 하도록 바꾼다 (환경변수 MUMC_EVAL_EVERY=N 이면 N epoch 마다 추가 평가).
# 또 학습 끝의 compute_vqa_acc 는 epoch 0..max_epoch-1 의 결과 파일을 모두 읽으므로, 없는 파일은 건너뛰게 vqaEvaluate.py 를 고친다.
_p5 = '/content/mumc/train_vqa.py'
_s5 = open(_p5).read()
_MARK5 = '# [patch] eval last epoch only'
if _MARK5 not in _s5 and '# [patch] val select' not in _s5:   # val 선택 패치가 이 마커를 대체하므로 함께 확인
    _old5 = ("            vqa_result = evaluation(model, test_loader, device, config)\n"
             "            json.dump(vqa_result, open(os.path.join(args.result_dir, '%s_vqa_result_%s.json' % (prefix, epoch)), 'w'))\n")
    assert _old5 in _s5, 'train_vqa.py 의 평가 줄을 찾지 못했습니다'
    _new5 = ("            " + _MARK5 + "\n"
             "            _ee = int(os.environ.get('MUMC_EVAL_EVERY', '0'))\n"
             "            if epoch == config['max_epoch'] - 1 or (_ee > 0 and (epoch + 1) % _ee == 0):\n"
             "                vqa_result = evaluation(model, test_loader, device, config)\n"
             "                json.dump(vqa_result, open(os.path.join(args.result_dir, '%s_vqa_result_%s.json' % (prefix, epoch)), 'w'))\n")
    open(_p5, 'w').write(_s5.replace(_old5, _new5, 1))
    print('train_vqa.py 평가 시점 패치 완료')
else:
    print('train_vqa.py 이미 패치됨')

_p6 = '/content/mumc/vqaEvaluate.py'
_s6 = open(_p6).read()
_MARK6 = '# [patch] skip missing'
if _MARK6 not in _s6:
    _old6 = "        resFile = res_file_path.replace('<epoch>', str(i))\n        print(resFile)\n"
    assert _old6 in _s6, 'vqaEvaluate.py 구조가 예상과 다릅니다'
    _new6 = _old6 + "        " + _MARK6 + "\n        if not os.path.exists(resFile):\n            continue\n"
    open(_p6, 'w').write(_s6.replace(_old6, _new6, 1))
    print('vqaEvaluate.py 패치 완료')
else:
    print('vqaEvaluate.py 이미 패치됨')

# train_vqa.py: val 로 epoch 를 고른다. 환경변수 MUMC_VAL_FILE 이 있으면 매 epoch val 을 평가해 *_val_result_<epoch>.json 으로 저장하고,
# val 정확도(pre_answer 완전일치, 정답 개수)가 최고인 epoch 의 가중치를 메모리(CPU)에 보관했다가, 학습이 끝난 뒤 그 epoch 의 test 를 한 번만 평가·저장한다.
# 마지막 epoch 의 test 도 함께 저장한다. 학습 도중에는 test 를 돌리지 않는다 (MUMC_EVAL_EVERY=N 을 주면 N epoch 마다 진단용으로 추가 저장).
_s7 = open(_p5).read()
_MARK7 = '# [patch] val select v2'
if _MARK7 not in _s7:
    _old_blk = ("            " + _MARK5 + "\n"
                "            _ee = int(os.environ.get('MUMC_EVAL_EVERY', '0'))\n"
                "            if epoch == config['max_epoch'] - 1 or (_ee > 0 and (epoch + 1) % _ee == 0):\n"
                "                vqa_result = evaluation(model, test_loader, device, config)\n"
                "                json.dump(vqa_result, open(os.path.join(args.result_dir, '%s_vqa_result_%s.json' % (prefix, epoch)), 'w'))\n")
    assert _old_blk in _s7, 'train_vqa.py 의 평가 블록(패치 후)을 찾지 못했습니다'
    _new_blk = ("            " + _MARK7 + "\n"
                "            _ee = int(os.environ.get('MUMC_EVAL_EVERY', '0'))\n"
                "            _do_test = epoch == config['max_epoch'] - 1 or (_ee > 0 and (epoch + 1) % _ee == 0)\n"
                "            if val_loader is not None:\n"
                "                val_result = evaluation(model, val_loader, device, config)\n"
                "                json.dump(val_result, open(os.path.join(args.result_dir, '%s_val_result_%s.json' % (prefix, epoch)), 'w'))\n"
                "                _vgt = {int(r['qid']): pre_answer(r['answer']) for r in val_loader.dataset.ann}\n"
                "                _ok = sum(pre_answer(r['answer']) == _vgt[int(r['qid'])] for r in val_result)\n"
                "                print('VAL epoch %d: correct %d / %d (best so far %d)' % (epoch, _ok, len(val_result), best_val_ok))\n"
                "                if _ok > best_val_ok:\n"
                "                    best_val_ok = _ok; best_epoch = epoch\n"
                "                    best_state = {k: v.detach().cpu().clone() for k, v in model_without_ddp.state_dict().items()}\n"
                "            if _do_test:\n"
                "                vqa_result = evaluation(model, test_loader, device, config)\n"
                "                json.dump(vqa_result, open(os.path.join(args.result_dir, '%s_vqa_result_%s.json' % (prefix, epoch)), 'w'))\n")
    _s7 = _s7.replace(_old_blk, _new_blk, 1)
    _old_loop = "    for epoch in range(start_epoch, config['max_epoch']):\n"
    assert _s7.count(_old_loop) == 1, 'epoch 루프를 찾지 못했습니다'
    _new_loop = ("    best_val_ok = -1; best_epoch = -1; best_state = None\n"
                 "    val_loader = None\n"
                 "    if os.environ.get('MUMC_VAL_FILE'):\n"
                 "        from torch.utils.data import DataLoader\n"
                 "        from dataset.vqa_dataset import vqa_dataset\n"
                 "        from dataset.utils import pre_answer\n"
                 "        _vd = vqa_dataset([os.environ['MUMC_VAL_FILE']], test_loader.dataset.transform, test_loader.dataset.vqa_root,\n"
                 "                          split='test', answer_list=config[args.dataset_use]['answer_list'])\n"
                 "        val_loader = DataLoader(_vd, batch_size=config['batch_size_test'], num_workers=4, shuffle=False)\n"
                 "        print('val dataset size: ', len(_vd))\n") + _old_loop
    _s7 = _s7.replace(_old_loop, _new_loop, 1)
    _old_end = "    total_time = time.time() - start_time\n"
    assert _s7.count(_old_end) == 1, 'train_vqa.py 의 학습 종료 줄을 찾지 못했습니다'
    _new_end = ("    if best_state is not None and best_epoch != config['max_epoch'] - 1:\n"
                "        model_without_ddp.load_state_dict(best_state)      # val 최고 epoch 의 가중치로 되돌려 test 를 한 번만 평가\n"
                "        print('val 최고 epoch %d 의 test 평가' % best_epoch)\n"
                "        vqa_result = evaluation(model_without_ddp, test_loader, device, config)\n"
                "        json.dump(vqa_result, open(os.path.join(args.result_dir, '%s_vqa_result_%s.json' % (prefix, best_epoch)), 'w'))\n") + _old_end
    _s7 = _s7.replace(_old_end, _new_end, 1)
    open(_p5, 'w').write(_s7)
    print('train_vqa.py val 선택 패치 완료')
else:
    print('train_vqa.py val 선택 이미 패치됨')


이미 패치됨
xbert.py 이미 패치됨
tokenization_bert.py 이미 패치됨
train_vqa.py 이미 패치됨
vqaEvaluate.py 이미 패치됨
train_vqa.py val 선택 이미 패치됨


In [ ]:
!cd /content/mumc && git checkout train_vqa.py vqaEvaluate.py

Updated 2 paths from the index


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/expert_cfg_mumc'
DATA_ROOT  = '/content/mumc_data'                 # VQA.yaml 의 /mnt/sda/lpf/data/vqa/ 를 이 경로로 치환
DATA_RAD_DIR   = f'{DATA_ROOT}/data_RAD'
DATA_SLAKE_DIR = f'{DATA_ROOT}/data_Slake'
PRETRAIN_DIR   = '/content/mumc/pretrain'
for d in [DRIVE_ROOT, DATA_RAD_DIR, f'{DATA_SLAKE_DIR}/en', PRETRAIN_DIR]:
    os.makedirs(d, exist_ok=True)
print('OK')

# VQA-RAD 분할
#   'public'  : OSF 공개 JSON  train 1,793 / test 451 (이전 방식)
#   'common'  : 공개 JSON trainset(3,064, frame 포함)에서 이미지 ID 단위로 val 을 뗀 공통 분할 — train 2,749 / val 315 / test 451
#   'merge'   : HF MERGE-Group/VQA-RAD (ExGra-Med 쪽 분할) — train 2,613 / val 451 / test 451. test 는 공통 분할과 같은 문항, train/val 경계만 다름
RAD_SPLIT = 'common'     # 'common': 팀 공통 분할(train 2,749 / val 315 / test 451), 'merge': MERGE-Group 분할. val 은 epoch 선택 전용, 학습에 쓰지 않음
COMMON_DIR = '/content/drive/MyDrive/vqarad_common'   # data_common/vqarad_img/{trainset,valset,testset}.json 을 올려 둘 Drive 폴더
# VQA-RAD 평가 프로토콜 (RAD_SPLIT 이 'common' 또는 'merge' 일 때)
#   'clean'    : 교수님 규칙 — train 만 학습, val 로 epoch 선택, 후보 목록 = train 정답만 (기본값)
#   'trainval' : train+val 로 학습, 후보 목록 = train+val 정답. val 선택이 없으므로 마지막 epoch 를 보고한다 (규칙 일부 완화)
#   'paper'    : train+val 로 학습, 후보 목록 = train+val+test 정답 (원 레포와 같은 구성). test 정답이 후보에 들어가므로 규칙 비준수 —
#                논문 수치와의 비교용으로만 쓰고, 결과표에는 반드시 별도 행으로 표기한다. test 문항 자체를 학습에 쓰지는 않는다.
RAD_PROTOCOL = 'paper'
assert RAD_PROTOCOL in ('clean', 'trainval', 'paper')
assert RAD_PROTOCOL == 'clean' or RAD_SPLIT in ('common', 'merge'), "trainval / paper 는 common 또는 merge 분할에서만 지원"
RUN_TAG = ''    # 곡선 진단용 재실행은 '_curve' (기존 결과를 덮어쓰지 않음)
RAD_OUT_NAME = {'common': 'rad_common', 'merge': 'rad_merge'}.get(RAD_SPLIT, 'rad')   # 결과 폴더를 분리해서 다른 분할의 epoch 결과가 섞이지 않게 한다
if RAD_PROTOCOL != 'clean':
    RAD_OUT_NAME += '_' + RAD_PROTOCOL      # 규칙 준수(clean) 결과와 폴더를 분리한다
RAD_OUT_NAME += RUN_TAG

Mounted at /content/drive
OK


## 1. 사전학습 체크포인트

In [19]:
import os, shutil

_ckpt_dst = os.path.join(PRETRAIN_DIR, 'med_pretrain_29.pth')
_drive_ckpt = f'{DRIVE_ROOT}/med_pretrain_29.pth'

if os.path.exists(_ckpt_dst):
    print('이미 존재:', _ckpt_dst)
elif os.path.exists(_drive_ckpt):
    shutil.copy2(_drive_ckpt, _ckpt_dst)
    print('Drive에서 복사 완료')
else:
    _FILE_ID = '1ZxwjfDeBYTMpw4mN_R9-gAcR9UOJiAFb'   # README 의 pretrained weights 링크
    !gdown {_FILE_ID} -O {_ckpt_dst}
    assert os.path.exists(_ckpt_dst) and os.path.getsize(_ckpt_dst) > 0, \
        '다운로드 실패 — README 링크에서 받아 Drive 에 올린 뒤 다시 실행하세요'
    shutil.copy2(_ckpt_dst, _drive_ckpt)
print(os.path.getsize(_ckpt_dst), 'bytes')

이미 존재: /content/mumc/pretrain/med_pretrain_29.pth
1818821353 bytes


## 2. 데이터 — VQA-RAD

`RAD_SPLIT='common'`(기본): 팀 공통 분할 **train 2,749 / val 315 / test 451** 을 Drive 의 `COMMON_DIR` 에서 읽어 `data_RAD/{trainset,valset,testset}.json` 으로 둡니다. val 은 epoch 선택에만 쓰고 학습에는 쓰지 않습니다. test 는 공개 JSON 의 451문항입니다.

`'merge'`: HF `MERGE-Group/VQA-RAD`(ExGra-Med 쪽 분할) **train 2,613 / val 451 / test 451**. test 451 은 공통 분할과 같은 문항이고 train/val 경계만 다릅니다(결과는 `output/rad_merge`). 질문 끝의 closed 옵션 문구와 `<image>` 토큰은 제거합니다.

`'public'` 은 이전 방식(OSF 공개 JSON 1,793/451)이고, 결과는 `output/rad` 에, common 은 `output/rad_common` 에 따로 저장됩니다.
정답 후보 목록은 common 에서는 **train 정답만**, public 에서는 레포 방식대로 train+test 의 모든 답을 `pre_answer` 로 정규화해 만듭니다.


In [8]:
import re
def pre_answer(answer):   # MUMC dataset/utils.py 의 pre_answer 와 동일
    answer = str(answer)
    answer = re.sub(r"([,.'!?\"()*#:;~])", '', answer.lower()).replace(' \t', ' ')
    answer = answer.replace('x ray', 'xray').replace('x-ray', 'xray')
    return answer.replace(' - ', '-')

import os, json, shutil, subprocess

_GDRIVE_JSON   = '/content/drive/MyDrive/vqa_rad/VQA_RAD Dataset Public.json'
_GDRIVE_IMGDIR = '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder'

def answer_type_of(ans):
    return 'CLOSED' if str(ans).strip().lower() in ('yes', 'no') else 'OPEN'

if RAD_SPLIT == 'common':
    train_data = json.load(open(f'{COMMON_DIR}/trainset.json'))
    val_data   = json.load(open(f'{COMMON_DIR}/valset.json'))
    test_data  = json.load(open(f'{COMMON_DIR}/testset.json'))
    for _r in train_data + val_data + test_data:
        _r['answer'] = str(_r['answer']); _r['answer_type'] = _r['answer_type'].strip().upper()
    assert (len(train_data), len(val_data), len(test_data)) == (2749, 315, 451), (len(train_data), len(val_data), len(test_data))
    all_data = train_data + val_data + test_data
    assert len({r['qid'] for r in all_data}) == len(all_data) == 3515, 'qid 중복 또는 총 문항 수 불일치'
    assert (sum(r['answer_type'] == 'CLOSED' for r in test_data), sum(r['answer_type'] == 'OPEN' for r in test_data)) == (272, 179), 'test closed/open 수가 다릅니다'
elif RAD_SPLIT == 'merge':
    # MERGE-Group/VQA-RAD (ExGra-Med 쪽이 쓰는 분할): train 2,613 / val 451 / test 451 (총 3,515, test 는 공통 분할의 test 와 같은 문항)
    # 파일의 질문에는 closed 옵션 문구와 <image> 토큰이 붙어 있어 제거한다 (다른 분할/모델과 같은 입력 조건)
    from huggingface_hub import hf_hub_download
    _OPT = re.compile(r'\s*\(Answer with either .*?\)\.?\s*(<image>)?\s*$', re.S)
    def _convert(fname):
        rows = json.load(open(hf_hub_download('MERGE-Group/VQA-RAD', fname, repo_type='dataset')))
        return [{'image_name': x['image'], 'qid': int(x['id']),
                 'question': _OPT.sub('', x['conversations'][0]['value'].replace('<image>', '')).strip(),
                 'answer': str(x['conversations'][1]['value']), 'answer_type': x['answer_type'].strip().upper(),
                 'question_type': 'unknown'} for x in rows]
    train_data, val_data, test_data = (_convert(f'{s}_w_options_new.json') for s in ('train', 'val', 'test'))
    assert (len(train_data), len(val_data), len(test_data)) == (2613, 451, 451), (len(train_data), len(val_data), len(test_data))
    all_data = train_data + val_data + test_data
    assert len({r['qid'] for r in all_data}) == len(all_data) == 3515, 'qid 중복 또는 총 문항 수 불일치'
    assert (sum(r['answer_type'] == 'CLOSED' for r in test_data), sum(r['answer_type'] == 'OPEN' for r in test_data)) == (272, 179), 'test closed/open 수가 다릅니다'
else:
    _raw_json = _GDRIVE_JSON
    if not os.path.exists(_raw_json):
        _raw_json = os.path.join(DATA_RAD_DIR, 'VQA_RAD_Dataset_Public.json')
        subprocess.run(['wget', '-q', '-O', _raw_json, 'https://osf.io/download/9dpze/'], timeout=120, check=True)

    with open(_raw_json) as f:
        all_data = json.load(f)
    for item in all_data:
        item['answer_type'] = item.get('answer_type', answer_type_of(item.get('answer', ''))).upper()
        item['answer'] = str(item['answer'])

    # qid 재번호: 원본 qid 에 정수/문자열이 섞여 있으면 pyarrow(M3AE) 변환이나 텐서 collate(MUMC)가 실패한다
    for _i, item in enumerate(all_data):
        item['qid'] = _i

    _TEST_TYPES = {'test_freeform', 'test_para'}
    _phrase_vals = {str(r.get('phrase_type', '')).lower() for r in all_data}
    assert _TEST_TYPES & _phrase_vals, 'phrase_type 에 test 표시가 없습니다 — 원본 JSON 을 확인하세요'
    test_data  = [r for r in all_data if str(r.get('phrase_type', '')).lower() in _TEST_TYPES]
    train_data = [r for r in all_data if str(r.get('phrase_type', '')).lower() not in _TEST_TYPES]
    assert len(test_data) == 451, len(test_data)
    assert len({r['qid'] for r in all_data}) == len(all_data), 'qid 가 중복됩니다'
    for r in all_data:
        assert 'image_name' in r and 'question_type' in r, '필수 키 누락(image_name/question_type): ' + str(sorted(r))


for r in all_data:
    assert 'image_name' in r and 'question_type' in r, '필수 키 누락(image_name/question_type): ' + str(sorted(r))

_fit_data = (train_data + val_data) if RAD_PROTOCOL != 'clean' else train_data      # trainval / paper 는 val 도 학습에 쓴다
with open(os.path.join(DATA_RAD_DIR, 'trainset.json'), 'w') as f: json.dump(_fit_data, f)
with open(os.path.join(DATA_RAD_DIR, 'testset.json'),  'w') as f: json.dump(test_data, f)
if RAD_SPLIT in ('common', 'merge'):
    with open(os.path.join(DATA_RAD_DIR, 'valset.json'), 'w') as f: json.dump(val_data, f)   # train_vqa.py 패치가 MUMC_VAL_FILE 로 읽는다

# 정답 후보 목록: 레포는 train+test 정답을 쓰지만(test 정답 문자열이 후보에 새어 들어감), common 은 train 정답만 쓴다
if RAD_SPLIT in ('common', 'merge'):
    _cand_src = {'clean': train_data, 'trainval': train_data + val_data, 'paper': train_data + val_data + test_data}[RAD_PROTOCOL]
else:
    _cand_src = train_data + test_data
if RAD_PROTOCOL != 'clean':
    print(f'※ RAD_PROTOCOL={RAD_PROTOCOL}: 학습 {len(_fit_data)}문항(train+val), 후보 목록 = ' + ('train+val 정답' if RAD_PROTOCOL == 'trainval' else 'train+val+test 정답 (규칙 비준수, 논문 비교용)'))
_answers = sorted({pre_answer(r['answer']) for r in _cand_src})
with open(os.path.join(DATA_RAD_DIR, 'answer_all_list.json'), 'w') as f: json.dump(_answers, f)
print(f'RAD_SPLIT={RAD_SPLIT}  train={len(train_data)}  test={len(test_data)}  answer_vocab={len(_answers)}')
print('test closed/open:', sum(r['answer_type'] == 'CLOSED' for r in test_data), sum(r['answer_type'] == 'OPEN' for r in test_data))
print('질문 예시:', [(r['question'][:50], r['answer'][:15]) for r in train_data[:3]])

_img_dst = os.path.join(DATA_RAD_DIR, 'images')
if not os.path.exists(_img_dst):
    assert os.path.isdir(_GDRIVE_IMGDIR), f'이미지 폴더 없음: {_GDRIVE_IMGDIR}'
    shutil.copytree(_GDRIVE_IMGDIR, _img_dst)
missing = [r['image_name'] for r in all_data if not os.path.exists(os.path.join(_img_dst, r['image_name']))]
assert not missing, f'이미지 {len(missing)}개 없음: {missing[:3]}'
print('images OK')

※ RAD_PROTOCOL=paper: 학습 3064문항(train+val), 후보 목록 = train+val+test 정답 (규칙 비준수, 논문 비교용)
RAD_SPLIT=common  train=2749  test=451  answer_vocab=509
test closed/open: 272 179
질문 예시: [('Are regions of the brain infarcted?', 'Yes'), ('Are the lungs normal appearing?', 'No'), ('Is there evidence of a pneumothorax', 'No')]
images OK


## 3. 데이터 — SLAKE

HF `BoKelvin/SLAKE` 의 원본 `train/validation/test.json` + `imgs.zip` 을 직접 받습니다
(`load_dataset` 표 형식에는 이미지 컬럼이 없음). 영어(`q_lang == 'en'`)만 쓰고,
MUMC 의 데이터 로더 키에 맞춰 `image_name`(= `img_name`, 예: `xmlab102/source.jpg`), `question_type`(= `content_type`)을 추가합니다.
이미지는 `imgs/xmlabN/source.jpg` 로 풀립니다. 답 후보는 레포 방식대로 **train+val** 입니다.

In [ ]:
import os, json, zipfile
from huggingface_hub import hf_hub_download

def convert(src, out_name):
    p = hf_hub_download('BoKelvin/SLAKE', src, repo_type='dataset')
    rows = [r for r in json.load(open(p)) if r['q_lang'] == 'en']
    for r in rows:
        r['image_name'] = r['img_name']
        r['question_type'] = r['content_type']
        r['answer'] = str(r['answer'])
        r['answer_type'] = r['answer_type'].upper()
    json.dump(rows, open(os.path.join(DATA_SLAKE_DIR, 'en', out_name), 'w'))
    n_open = sum(r['answer_type'] == 'OPEN' for r in rows)
    print(f'{src} -> en/{out_name}: {len(rows)}  (OPEN {n_open} / CLOSED {len(rows) - n_open})')
    return rows

slake_train = convert('train.json', 'slake_train.json')
slake_val   = convert('validation.json', 'slake_val.json')
slake_test  = convert('test.json', 'slake_test.json')     # 기대값: 1061 = OPEN 645 / CLOSED 416

# 정답 후보 목록은 train 정답만 (val 은 epoch 선택에만 쓰고 학습·후보에 넣지 않는다)
_answers = sorted({pre_answer(r['answer']) for r in slake_train})
json.dump(_answers, open(os.path.join(DATA_SLAKE_DIR, 'en', 'answer_train_list.json'), 'w'))
_vocab = set(_answers)
_unseen = sum(pre_answer(r['answer']) not in _vocab for r in slake_test)
print('answer_vocab =', len(_answers), '| test 중 후보에 없는 정답 수(정확도 상한을 낮춤):', _unseen)

if not os.path.isdir(os.path.join(DATA_SLAKE_DIR, 'imgs')):
    _zip = hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')
    with zipfile.ZipFile(_zip) as z:
        z.extractall(DATA_SLAKE_DIR)
assert all(os.path.exists(os.path.join(DATA_SLAKE_DIR, 'imgs', r['image_name'])) for r in slake_train + slake_val + slake_test), '이미지 누락'
print('images OK')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

train.json -> en/slake_train.json: 4919  (OPEN 2976 / CLOSED 1943)


validation.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

validation.json -> en/slake_val.json: 1053  (OPEN 631 / CLOSED 422)


test.json:   0%|          | 0.00/636k [00:00<?, ?B/s]

test.json -> en/slake_test.json: 1061  (OPEN 645 / CLOSED 416)
answer_vocab = 221 | test 중 후보에 없는 정답 수(정확도 상한을 낮춤): 2


imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

images OK


In [ ]:
# SLAKE 분할 점검 (영어만): train/val/test 가 이미지 단위로 분리돼 있는지, 같은 문항이 겹치는지, 정답이 train 정답 집합에 있는 비율(candidate coverage)을 본다.
import json
from huggingface_hub import hf_hub_download
S = {k: [r for r in json.load(open(hf_hub_download('BoKelvin/SLAKE', f, repo_type='dataset'), encoding='utf8')) if r.get('q_lang', 'en') == 'en']
     for k, f in (('train', 'train.json'), ('val', 'validation.json'), ('test', 'test.json'))}
for k, v in S.items():
    print(f"{k:5s} 문항 {len(v):5d}  이미지 {len({r['img_name'] for r in v}):4d}  closed {sum(r['answer_type'] == 'CLOSED' for r in v)} / open {sum(r['answer_type'] == 'OPEN' for r in v)}")
im = {k: {r['img_name'] for r in v} for k, v in S.items()}
key = lambda r: (r['img_name'], r['question'].strip().lower(), str(r['answer']).strip().lower())
for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    print(f"{a}∩{b}  이미지 겹침 {len(im[a] & im[b])}  |  동일 문항(이미지+질문+답) {len({key(r) for r in S[a]} & {key(r) for r in S[b]})}")
assert not (im['train'] & im['val']), 'train 과 val 이 이미지를 공유합니다'
_tr = {str(r['answer']).strip().lower() for r in S['train']}
for k in ('val', 'test'):
    _cov = lambda rows: 100 * sum(str(r['answer']).strip().lower() in _tr for r in rows) / len(rows)
    print(f"{k:4s} 정답이 train 정답 집합에 있는 비율: overall {_cov(S[k]):.1f}%  open {_cov([r for r in S[k] if r['answer_type'] == 'OPEN']):.1f}%  closed {_cov([r for r in S[k] if r['answer_type'] == 'CLOSED']):.1f}%")


train 문항  4919  이미지  450  closed 1943 / open 2976
val   문항  1053  이미지   96  closed 422 / open 631
test  문항  1061  이미지   96  closed 416 / open 645
train∩val  이미지 겹침 0  |  동일 문항(이미지+질문+답) 0
train∩test  이미지 겹침 0  |  동일 문항(이미지+질문+답) 0
val∩test  이미지 겹침 0  |  동일 문항(이미지+질문+답) 0
val  정답이 train 정답 집합에 있는 비율: overall 99.7%  open 99.5%  closed 100.0%
test 정답이 train 정답 집합에 있는 비율: overall 99.8%  open 99.7%  closed 100.0%


## 4. 설정 패치 + 학습 — VQA-RAD

`VQA.yaml` 의 경로 접두어를 치환하고(문자열 치환 후 `grep` 으로 확인), 학습합니다.
매 epoch 마다 test 예측이 `$MUMC_OUT_DIR/result/` 에 저장됩니다. 동작 확인용으로 `MAX_EPOCH=1` 로 먼저 돌려보세요 (기본 40).

In [9]:
import re
MAX_EPOCH = 40     # 스모크 테스트는 1
EVAL_EVERY = 5      # val 은 매 epoch 평가하고, test 는 마지막 epoch 와 val 최고 epoch 에서만(학습 종료 후) 평가·저장한다. N>0 이면 N epoch 마다 진단용으로 추가 저장

_y = '/content/mumc/configs/VQA.yaml'
_t = open(_y).read()
if '/mnt/sda/lpf/data/vqa/' in _t:
    _t = _t.replace('/mnt/sda/lpf/data/vqa/', DATA_ROOT + '/')
_t, n = re.subn(r'^max_epoch:\s*\d+', f'max_epoch: {MAX_EPOCH}', _t, flags=re.M)
assert n == 1, 'max_epoch 줄을 찾지 못했습니다'
# SLAKE: 레포 설정은 slake_train + slake_val 로 학습한다 -> train 만 쓰도록 바꾸고, 정답 후보도 train 만 쓴다
_t = re.sub(r"(slake_train\.json')\s*,\s*'[^']*slake_val\.json'", r"\1", _t)
_t = _t.replace('answer_trainval_list.json', 'answer_train_list.json')
assert 'slake_val' not in _t and 'answer_train_list.json' in _t, 'SLAKE 설정 패치 실패'
assert '/mnt/sda' not in _t
open(_y, 'w').write(_t)
!grep -n "file\|answer_list\|vqa_root\|max_epoch\|init_lr" /content/mumc/configs/VQA.yaml | head -30

2:  train_file: [ '/content/mumc_data/data_RAD/trainset.json' ]
3:  test_file: [ '/content/mumc_data/data_RAD/testset.json' ]
4:  answer_list: '/content/mumc_data/data_RAD/answer_all_list.json'
5:  vqa_root: '/content/mumc_data/data_RAD/images/'  # vqa_rad
7:  train_file: [ '/content/mumc_data/data_PathVQA/pathvqa_train.json',
9:  test_file: [ '/content/mumc_data/data_PathVQA/pathvqa_test.json' ]
10:  answer_list: '/content/mumc_data/data_PathVQA/answer_train_list.json'
11:  vqa_root: '/content/mumc_data/data_PathVQA/images/'
13:  train_file: [ '/content/mumc_data/data_Slake/en/slake_train.json' ]
14:  test_file: [ '/content/mumc_data/data_Slake/en/slake_test.json' ]
15:  answer_list: '/content/mumc_data/data_Slake/en/answer_train_list.json'
16:  vqa_root: '/content/mumc_data/data_Slake/imgs/'  # slake
18:  train_file: [ '/content/mumc_data/vqa_med/2019/vqa_med_2019_train.json',
20:  test_file: [ '/content/mumc_data/vqa_med/2019/vqa_med_2019_test.json' ]
21:  answer_list: '/content/mum

In [ ]:
# import 스모크 테스트 — 환경이 안 맞으면 여기서 멈춘다 (학습 전에 확인)
%cd /content/mumc
!python3 -c "import ruamel.yaml, timm, transformers, cv2, skimage; from models.model_vqa import MUMC_VQA; from dataset import create_dataset; print('import OK', transformers.__version__)"

/content
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
import OK 4.43.0


In [10]:
%cd /content/mumc
OUT_RAD = f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}'
VAL_ENV = f'MUMC_VAL_FILE={DATA_RAD_DIR}/valset.json' if RAD_PROTOCOL == 'clean' else ''   # trainval / paper: val 이 학습에 들어가므로 val 선택을 하지 않는다 (마지막 epoch 평가)
!MUMC_OUT_DIR={OUT_RAD} {VAL_ENV} MUMC_EVAL_EVERY={EVAL_EVERY} python3 train_vqa.py \
    --dataset_use rad \
    --checkpoint {PRETRAIN_DIR}/med_pretrain_29.pth

/content
config:  {'rad': {'train_file': ['/content/mumc_data/data_RAD/trainset.json'], 'test_file': ['/content/mumc_data/data_RAD/testset.json'], 'answer_list': '/content/mumc_data/data_RAD/answer_all_list.json', 'vqa_root': '/content/mumc_data/data_RAD/images/'}, 'pathvqa': {'train_file': ['/content/mumc_data/data_PathVQA/pathvqa_train.json', '/content/mumc_data/data_PathVQA/pathvqa_val.json'], 'test_file': ['/content/mumc_data/data_PathVQA/pathvqa_test.json'], 'answer_list': '/content/mumc_data/data_PathVQA/answer_train_list.json', 'vqa_root': '/content/mumc_data/data_PathVQA/images/'}, 'slake': {'train_file': ['/content/mumc_data/data_Slake/en/slake_train.json'], 'test_file': ['/content/mumc_data/data_Slake/en/slake_test.json'], 'answer_list': '/content/mumc_data/data_Slake/en/answer_train_list.json', 'vqa_root': '/content/mumc_data/data_Slake/imgs/'}, 'med2019': {'train_file': ['/content/mumc_data/vqa_med/2019/vqa_med_2019_train.json', '/content/mumc_data/vqa_med/2019/vqa_med_2019

## 5. 학습 — SLAKE

In [ ]:
%cd /content/mumc

OUT_SLAKE = f'{DRIVE_ROOT}/output/slake_val_sel'   # train 만 학습 + val 로 epoch 선택 (이전 train+val 학습 결과 'slake' 와 분리)
!MUMC_OUT_DIR={OUT_SLAKE} MUMC_VAL_FILE={DATA_SLAKE_DIR}/en/slake_val.json MUMC_EVAL_EVERY={EVAL_EVERY} python3 train_vqa.py \
    --dataset_use slake \
    --checkpoint {PRETRAIN_DIR}/med_pretrain_29.pth

/content
config:  {'rad': {'train_file': ['/content/mumc_data/data_RAD/trainset.json'], 'test_file': ['/content/mumc_data/data_RAD/testset.json'], 'answer_list': '/content/mumc_data/data_RAD/answer_all_list.json', 'vqa_root': '/content/mumc_data/data_RAD/images/'}, 'pathvqa': {'train_file': ['/content/mumc_data/data_PathVQA/pathvqa_train.json', '/content/mumc_data/data_PathVQA/pathvqa_val.json'], 'test_file': ['/content/mumc_data/data_PathVQA/pathvqa_test.json'], 'answer_list': '/content/mumc_data/data_PathVQA/answer_train_list.json', 'vqa_root': '/content/mumc_data/data_PathVQA/images/'}, 'slake': {'train_file': ['/content/mumc_data/data_Slake/en/slake_train.json'], 'test_file': ['/content/mumc_data/data_Slake/en/slake_test.json'], 'answer_list': '/content/mumc_data/data_Slake/en/answer_train_list.json', 'vqa_root': '/content/mumc_data/data_Slake/imgs/'}, 'med2019': {'train_file': ['/content/mumc_data/vqa_med/2019/vqa_med_2019_train.json', '/content/mumc_data/vqa_med/2019/vqa_med_2019

## 6. 결과 — Phi-3.5 노트북과 같은 지표로 재채점

`result/*_vqa_result_<epoch>.json` 의 예측(`qid`, `answer`)을 test 정답과 비교해
**closed accuracy / open recall / overall** 을 계산합니다. val 셋이 없어서 `best epoch` 는 test 로 고른 값이므로
**주 결과는 마지막 epoch**, best 는 참고(낙관적)입니다. 레포 자체 지표(exact match)는 `result/all_acc.json` 에 있습니다.

In [12]:
import re, glob, json, os
import numpy as np
from collections import Counter

# 채점은 기존 방식 그대로: closed = 정답 단어가 예측 토큰에 포함되면 정답, open = 토큰 recall
def normalize(text):
    return re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()

def closed_accuracy(pred, gt):
    return float(str(gt).strip().lower() in normalize(pred))

def open_recall(pred, gt):
    gt_toks, pred_toks = Counter(normalize(gt)), Counter(normalize(pred))
    if not gt_toks:
        return 0.0
    return sum((gt_toks & pred_toks).values()) / sum(gt_toks.values())

def score_result(test_file, result_file):
    gts = {r['qid']: r for r in json.load(open(test_file))}
    preds = {r['qid']: r['answer'] for r in json.load(open(result_file))}
    assert set(preds) == set(gts), '예측 qid 가 test 와 다릅니다'
    cl, op = [], []
    for q, r in gts.items():
        if r['answer_type'] == 'CLOSED':
            cl.append(closed_accuracy(preds[q], r['answer']))
        else:
            op.append(open_recall(preds[q], r['answer']))
    return {'open': 100 * np.mean(op) if op else float('nan'), 'closed': 100 * np.mean(cl) if cl else float('nan'),
            'overall': 100 * np.mean(cl + op), 'n_open': len(op), 'n_closed': len(cl)}

def _epochs(out_dir, kind):    # kind: 'vqa' (test 결과) | 'val' (val 결과)
    pat = re.compile(rf'_{kind}_result_(\d+)\.json$')
    return {int(pat.search(f).group(1)): f for f in glob.glob(os.path.join(out_dir, 'result', '*.json')) if pat.search(f)}

def select_by_val(out_dir, val_file):
    """학습 중 쓴 기준과 같다: val 에서 pre_answer 완전일치 개수가 가장 많은 epoch (동점이면 이른 epoch). test 는 보지 않는다."""
    vgt = {int(r['qid']): pre_answer(r['answer']) for r in json.load(open(val_file))}
    vals = _epochs(out_dir, 'val')
    assert vals, f'val 결과 파일이 없습니다: {out_dir}/result'
    ok = {ep: sum(pre_answer(r['answer']) == vgt[int(r['qid'])] for r in json.load(open(f))) for ep, f in vals.items()}
    return max(sorted(ok), key=lambda e: (ok[e], -e)), ok

def evaluate_run(out_dir, val_file, test_file):
    tests = _epochs(out_dir, 'vqa')
    if not _epochs(out_dir, 'val'):      # val 선택 없이 학습한 프로토콜(trainval / paper): 마지막 epoch 를 보고값으로 쓴다
        _last = max(tests); _r = score_result(test_file, tests[_last])
        return {'val-selected': (_last, _r), 'last': (_last, _r)}, {}
    sel, ok = select_by_val(out_dir, val_file)
    assert sel in tests, f'선택된 epoch {sel} 의 test 결과가 없습니다 (학습 중 패치가 적용됐는지 확인)'
    last = max(tests)
    return {'val-selected': (sel, score_result(test_file, tests[sel])), 'last': (last, score_result(test_file, tests[last]))}, ok

paper = {'VQA-RAD': {'open': 71.50, 'closed': 84.20, 'overall': 79.20},
         'SLAKE':   {'open': None,  'closed': None,  'overall': 84.90}}
RUNS = [('VQA-RAD', f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}', f'{DATA_RAD_DIR}/valset.json', f'{DATA_RAD_DIR}/testset.json'),
        ('SLAKE',   f'{DRIVE_ROOT}/output/slake_val_sel', f'{DATA_SLAKE_DIR}/en/slake_val.json', f'{DATA_SLAKE_DIR}/en/slake_test.json')]
if RAD_PROTOCOL != 'clean':
    print(f'※ RAD_PROTOCOL={RAD_PROTOCOL}: 규칙 준수 결과가 아닙니다. VQA-RAD 행은 val 선택이 아니라 마지막 epoch 이며, 논문 비교용으로만 별도 표기하세요.\n')
results = {}
for ds, out_dir, vf, tf in RUNS:
    try:
        results[ds], ok = evaluate_run(out_dir, vf, tf)
    except (AssertionError, OSError) as e:
        print(f'[{ds}] 결과 없음:', e); continue
    p = paper[ds]
    print(f'[{ds}] paper open={p["open"]} closed={p["closed"]} overall={p["overall"]}')
    print(f'  val 정답 수 by epoch: {dict(sorted(ok.items()))}')
    for k, (ep, m) in results[ds].items():
        print(f'  {k:14s} epoch={ep:2d}  open={m["open"]:6.2f}  closed={m["closed"]:6.2f}  overall={m["overall"]:6.2f}  (n_open={m["n_open"]}, n_closed={m["n_closed"]})')

# ── 논문 수치와 비교 (val 로 고른 epoch 이 주 결과) ──
print(f"\n{'dataset':8s} {'metric':8s} {'ours':>7s} {'paper':>7s} {'diff':>7s}")
for ds in ('VQA-RAD', 'SLAKE'):
    if ds not in results:
        print(f'{ds:8s} (결과 없음)'); continue
    m = results[ds]['val-selected'][1]
    for k in ('open', 'closed', 'overall'):
        pv = paper[ds][k]
        print(f'{ds:8s} {k:8s} {m[k]:7.2f} ' + (f'{pv:7.2f} {m[k] - pv:+7.2f}' if pv is not None else '      —       —'))

# ── candidate coverage (VQA-RAD): test 정답이 train 전용 후보 목록 안에 있는 비율 ──
# 후보에 없는 test 문항은 지우지 않고 그대로 채점한다 (정답을 낼 수 없으므로 0점). 제거하면 어려운 문항이 사라져 점수가 부풀기 때문.
if 'VQA-RAD' in results:
    assert RAD_SPLIT in ('common', 'merge'), '후보 목록이 train 전용인 분할에서만 의미가 있습니다'
    _cands = set(json.load(open(f'{DATA_RAD_DIR}/answer_all_list.json')))
    _tr = json.load(open(f'{DATA_RAD_DIR}/trainset.json'))      # clean: train, 그 외: train+val
    _exp = _tr + (json.load(open(f'{DATA_RAD_DIR}/testset.json')) if RAD_PROTOCOL == 'paper' else [])
    assert _cands == {pre_answer(r['answer']) for r in _exp}, f'후보 목록이 프로토콜({RAD_PROTOCOL})의 구성과 다릅니다'
    def _cov(path):
        rows = json.load(open(path)); out = {}
        for name, sel in (('overall', rows), ('open', [r for r in rows if r['answer_type'] == 'OPEN']), ('closed', [r for r in rows if r['answer_type'] == 'CLOSED'])):
            hit = sum(pre_answer(r['answer']) in _cands for r in sel)
            out[name] = {'in_candidates': hit, 'n': len(sel), 'coverage': round(100 * hit / len(sel), 2)}
        return out
    cov = {'test': _cov(f'{DATA_RAD_DIR}/testset.json'), 'val': _cov(f'{DATA_RAD_DIR}/valset.json'), 'n_candidates': len(_cands)}
    n_test_scored = results['VQA-RAD']['val-selected'][1]['n_open'] + results['VQA-RAD']['val-selected'][1]['n_closed']
    assert n_test_scored == cov['test']['overall']['n'], 'test 문항이 채점에서 빠졌습니다'
    _desc = {"clean": "train 정답만", "trainval": "train+val 정답", "paper": "train+val+test 정답 — test 정답 포함!"}[RAD_PROTOCOL]
    print(f'\n[candidate coverage] 후보 {len(_cands)}개 ({_desc})')
    for sp in ('test', 'val'):
        c = cov[sp]
        print(f"  {sp:4s} overall {c['overall']['coverage']:6.2f}% ({c['overall']['in_candidates']}/{c['overall']['n']}) | open {c['open']['coverage']:6.2f}% ({c['open']['in_candidates']}/{c['open']['n']}) | closed {c['closed']['coverage']:6.2f}% ({c['closed']['in_candidates']}/{c['closed']['n']})")
    print(f'  test {n_test_scored}문항 전부 채점됨 (후보에 없는 문항도 제거하지 않음)')
    json.dump(cov, open(f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}/candidate_coverage.json', 'w'), indent=1)


※ RAD_PROTOCOL=paper: 규칙 준수 결과가 아닙니다. VQA-RAD 행은 val 선택이 아니라 마지막 epoch 이며, 논문 비교용으로만 별도 표기하세요.

[VQA-RAD] paper open=71.5 closed=84.2 overall=79.2
  val 정답 수 by epoch: {}
  val-selected   epoch=39  open= 65.50  closed= 80.88  overall= 74.78  (n_open=179, n_closed=272)
  last           epoch=39  open= 65.50  closed= 80.88  overall= 74.78  (n_open=179, n_closed=272)
[SLAKE] 결과 없음: [Errno 2] No such file or directory: '/content/mumc_data/data_Slake/en/slake_val.json'

dataset  metric      ours   paper    diff
VQA-RAD  open       65.50   71.50   -6.00
VQA-RAD  closed     80.88   84.20   -3.32
VQA-RAD  overall    74.78   79.20   -4.42
SLAKE    (결과 없음)

[candidate coverage] 후보 509개 (train+val+test 정답 — test 정답 포함!)
  test overall 100.00% (451/451) | open 100.00% (179/179) | closed 100.00% (272/272)
  val  overall 100.00% (315/315) | open 100.00% (108/108) | closed 100.00% (207/207)
  test 451문항 전부 채점됨 (후보에 없는 문항도 제거하지 않음)


In [13]:
# 8-2. 저장된 test 결과를 epoch 별로 모두 채점한다 (진단용 — val 로 고른 값이 아니므로 보고값으로 쓰지 말 것). 결과 셀(채점 함수 정의)을 먼저 실행해야 한다.
_out = f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}'
_tests = _epochs(_out, 'vqa')
print(f'{_out}\n저장된 test 결과 epoch: {sorted(_tests)}')
print(f"{'epoch':>6} {'open':>7} {'closed':>7} {'overall':>8}")
_rows = {}
for ep in sorted(_tests):
    m = score_result(f'{DATA_RAD_DIR}/testset.json', _tests[ep]); _rows[ep] = m
    print(f"{ep:6d} {m['open']:7.2f} {m['closed']:7.2f} {m['overall']:8.2f}")
_best = max(_rows, key=lambda e: _rows[e]['overall'])
print(f"\n저장된 epoch 중 test overall 최고: epoch {_best} ({_rows[_best]['overall']:.2f}) — test 로 고른 값이라 보고 불가, epoch 간 변동 폭을 보는 용도")
print(f"overall 최소~최대: {min(r['overall'] for r in _rows.values()):.2f} ~ {max(r['overall'] for r in _rows.values()):.2f}")


/content/drive/MyDrive/expert_cfg_mumc/output/rad_common_paper
저장된 test 결과 epoch: [4, 9, 14, 19, 24, 29, 34, 39]
 epoch    open  closed  overall
     4   35.65   74.63    59.16
     9   60.61   79.41    71.95
    14   63.85   80.51    73.90
    19   61.98   79.78    72.72
    24   64.66   81.25    74.67
    29   64.85   80.51    74.30
    34   64.57   81.25    74.63
    39   65.50   80.88    74.78

저장된 epoch 중 test overall 최고: epoch 39 (74.78) — test 로 고른 값이라 보고 불가, epoch 간 변동 폭을 보는 용도
overall 최소~최대: 59.16 ~ 74.78


In [20]:
# 8-3. MUMC 가 틀린 VQA-RAD test 문항 보기. 결과 셀(채점 함수 정의)을 먼저 실행해야 한다. 진단용이며 보고값을 바꾸지 않는다.
import json, re, os
from collections import Counter
EPOCH = None            # None 이면 저장된 마지막 epoch. 다른 epoch 를 보려면 숫자를 넣는다 (예: 29)
_out = f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}'
_tests = _epochs(_out, 'vqa'); ep = max(_tests) if EPOCH is None else EPOCH
test = json.load(open(f'{DATA_RAD_DIR}/testset.json'))
gts  = {int(r['qid']): r for r in test}
def load(path): return {int(r['qid']): r['answer'] for r in json.load(open(path))}
def per_q(preds):
    out = {}
    for q, r in gts.items():
        pr, gt = str(preds[q]), str(r['answer'])
        if r['answer_type'] == 'CLOSED':
            out[q] = closed_accuracy(pr, gt)
        else:
            out[q] = open_recall(pr, gt)
    return out
P = load(_tests[ep]); S = per_q(P)
wrong = [q for q, s in S.items() if s < 1]
nc = sum(gts[q]['answer_type'] == 'CLOSED' for q in wrong); no = len(wrong) - nc
print(f'{RAD_OUT_NAME} epoch {ep}: 점수<1 문항 {len(wrong)}개 (CLOSED {nc} / OPEN {no})')

# closed: yes/no 혼동
cl = [q for q in gts if gts[q]['answer_type'] == 'CLOSED']
conf = Counter((str(gts[q]['answer']).strip().lower(), (normalize(P[q]) or [''])[0]) for q in cl if str(gts[q]['answer']).strip().lower() in ('yes', 'no'))
print('[CLOSED] 정답->예측(첫 단어):', dict(conf))
print('\n[CLOSED 오답 예시 20]')
for q in [q for q in wrong if gts[q]['answer_type'] == 'CLOSED'][:20]:
    print(f"  gt={str(gts[q]['answer'])[:22]!r:26} pred={str(P[q])[:22]!r:26} Q: {gts[q]['question'][:70]}")
print('\n[OPEN 오답 예시 25 — 점수 낮은 순]')
for q in sorted([q for q in wrong if gts[q]['answer_type'] == 'OPEN'], key=lambda q: S[q])[:25]:
    print(f"  {S[q]:.2f} gt={str(gts[q]['answer'])[:22]!r:26} pred={str(P[q])[:22]!r:26} Q: {gts[q]['question'][:60]}")

# 규칙 준수(clean) 실행과 비교: 한쪽만 맞은 문항 수
_clean = f'{DRIVE_ROOT}/output/' + RAD_OUT_NAME.replace('_paper', '').replace('_trainval', '')
if _clean != _out and os.path.isdir(_clean) and _epochs(_clean, 'vqa'):
    try:
        cep = select_by_val(_clean, f'{DATA_RAD_DIR}/valset.json')[0]
    except (AssertionError, OSError):
        cep = max(_epochs(_clean, 'vqa'))
    S2 = per_q(load(_epochs(_clean, 'vqa')[cep]))
    both_ok = sum(S[q] == 1 and S2[q] == 1 for q in gts); both_bad = sum(S[q] < 1 and S2[q] < 1 for q in gts)
    only_new = [q for q in gts if S[q] == 1 and S2[q] < 1]; only_old = [q for q in gts if S[q] < 1 and S2[q] == 1]
    print(f'\n[clean(epoch {cep}) vs 이번 실행(epoch {ep})] 둘 다 정답 {both_ok} | 둘 다 오답 {both_bad} | 이번만 정답 {len(only_new)} | clean 만 정답 {len(only_old)}')
    print('  (closed 가 내려간 원인 후보: clean 만 정답인 CLOSED 문항 수 =', sum(gts[q]['answer_type'] == 'CLOSED' for q in only_old), ')')
    print('\n[clean 만 맞힌 문항 예시 15]')
    for q in only_old[:15]:
        print(f"  {gts[q]['answer_type'][:2]} gt={str(gts[q]['answer'])[:20]!r:24} 이번={str(P[q])[:20]!r:24} Q: {gts[q]['question'][:55]}")
else:
    print('\n(clean 실행 폴더를 찾지 못해 비교는 생략)')
json.dump([{'qid': q, 'type': gts[q]['answer_type'], 'score': S[q], 'question': gts[q]['question'], 'gt': gts[q]['answer'], 'pred': P[q]} for q in wrong],
          open(f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}_wrong_ep{ep}.json', 'w'), ensure_ascii=False, indent=1)
print('\n전체 오답 목록 저장:', f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}_wrong_ep{ep}.json')


rad_common_paper epoch 39: 점수<1 문항 120개 (CLOSED 52 / OPEN 68)
[CLOSED] 정답->예측(첫 단어): {('yes', 'no'): 16, ('yes', 'yes'): 101, ('no', 'no'): 110, ('no', 'yes'): 23, ('yes', 'hemorrhage'): 1}

[CLOSED 오답 예시 20]
  gt='yes'                      pred='no'                       Q: Is there evidence of an aortic aneurysm?
  gt='Yes'                      pred='no'                       Q: Is there airspace consolidation on the left side?
  gt='yes'                      pred='no'                       Q: Is there vascular pathology in this image?
  gt='Yes'                      pred='no'                       Q: is the cardiac silhouette small/stretched
  gt='Yes'                      pred='no'                       Q: Is the cardiac silhouette less than half the lateral diameter of the c
  gt='Yes'                      pred='no'                       Q: Is the outline of visceral pleura visible in the upper left lung field
  gt='No'                       pred='yes'                      Q: Is t

In [23]:
# 8-4. 채점 함수 비교 — 현재(원본) 채점 v1 과, 표기 차이(하이픈·관사)를 흡수하는 v2. 진단용이며 어느 쪽을 공식으로 쓸지는 정하기 전이다.
# v1 : 소문자 + 영숫자 외 문자를 공백으로 바꾸고 단어로 나눔 (x-ray -> x, ray / xray -> xray 로 달라진다)
# v2 : v1 보다 낮아지지 않는 상위 호환 채점. 아래 셋 중 가장 높은 값을 쓴다
#      (a) v1 점수  (b) 하이픈·아포스트로피를 붙이고 관사(a/an/the)를 뺀 토큰 recall  (c) 공백·구두점을 모두 지운 문자열이 완전히 같으면 1점 (x-ray = x ray = xray)
# 결과 셀(채점 함수 정의)을 먼저 실행해야 한다.
import json, re, os
from collections import Counter

def _protect(s):      # 숫자 사이의 소수점은 지우지 않는다 (3.4 와 34 는 다른 값)
    return re.sub(r'(?<=\d)\.(?=\d)', 'dot', str(s).lower())
def norm_v2(text):
    t = _protect(text).replace('-', '').replace("'", '')
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return [w for w in t.split() if w not in ('a', 'an', 'the')]
def squeeze(s): return re.sub(r'[^a-z0-9]', '', _protect(s))
def closed_v2(pred, gt):
    g = str(gt).strip().lower()
    return float(g in normalize(pred) or g in norm_v2(pred) or (squeeze(pred) == squeeze(gt) and squeeze(gt) != ''))
def open_v2(pred, gt):
    g, p = Counter(norm_v2(gt)), Counter(norm_v2(pred))
    r2 = sum((g & p).values()) / sum(g.values()) if g else 0.0
    ex = 1.0 if squeeze(gt) != '' and squeeze(pred) == squeeze(gt) else 0.0
    return max(open_recall(pred, gt), r2, ex)

test = json.load(open(f'{DATA_RAD_DIR}/testset.json')); gts = {int(r['qid']): r for r in test}
def load(path): return {int(r['qid']): r['answer'] for r in json.load(open(path))}
def both(preds):
    s1, s2 = {}, {}
    for q, r in gts.items():
        pr, gt = preds[q], r['answer']
        if r['answer_type'] == 'CLOSED': s1[q], s2[q] = closed_accuracy(pr, gt), closed_v2(pr, gt)
        else: s1[q], s2[q] = open_recall(pr, gt), open_v2(pr, gt)
    return s1, s2
def summ(s):
    op = [s[q] for q in s if gts[q]['answer_type'] == 'OPEN']; cl = [s[q] for q in s if gts[q]['answer_type'] == 'CLOSED']
    return 100 * sum(op) / len(op), 100 * sum(cl) / len(cl), 100 * (sum(op) + sum(cl)) / (len(op) + len(cl))

runs = []
_pd = f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}'
runs.append((f'{RAD_OUT_NAME} (마지막 epoch)', _epochs(_pd, 'vqa')[max(_epochs(_pd, 'vqa'))]))
_cd = f'{DRIVE_ROOT}/output/' + RAD_OUT_NAME.replace('_paper', '').replace('_trainval', '')
if _cd != _pd and os.path.isdir(_cd) and _epochs(_cd, 'vqa'):
    try: cep = select_by_val(_cd, f'{DATA_RAD_DIR}/valset.json')[0]
    except (AssertionError, OSError): cep = max(_epochs(_cd, 'vqa'))
    runs.append((f'clean (val 선택 epoch {cep})', _epochs(_cd, 'vqa')[cep]))
print(f"{'실행':34s} {'채점':4s} {'open':>7s} {'closed':>7s} {'overall':>8s}")
for name, f in runs:
    s1, s2 = both(load(f))
    for tag, s in (('v1', s1), ('v2', s2)):
        o, c, a = summ(s); print(f'{name:34s} {tag:4s} {o:7.2f} {c:7.2f} {a:8.2f}')
    ch = [q for q in gts if abs(s1[q] - s2[q]) > 1e-9]
    print(f'  -> v1 과 점수가 다른 문항 {len(ch)}개 (open {sum(gts[q]["answer_type"] == "OPEN" for q in ch)} / closed {len(ch) - sum(gts[q]["answer_type"] == "OPEN" for q in ch)})')
    P = load(f)
    for q in ch[:20]: print(f"     {s1[q]:.2f}->{s2[q]:.2f} gt={str(gts[q]['answer'])[:24]!r:28} pred={str(P[q])[:24]!r}")

# 논문 수치를 낸 원 레포의 평가 코드가 어떤 정규화를 하는지 확인한다 (어느 채점이 논문과 같은 조건인지 판단하는 근거)
print('\n===== /content/mumc/vqaEvaluate.py (정규화·정확도 계산 부분) =====')
_s = open('/content/mumc/vqaEvaluate.py').read()
for kw in ('def ', 'punct', 'contraction', 'article', 'accuracy', 'answer_type', 'ansType'):
    pass
print(_s[:3500])


실행                                 채점      open  closed  overall
rad_common_paper (마지막 epoch)       v1     65.50   80.88    74.78
rad_common_paper (마지막 epoch)       v2     66.43   81.25    75.37
  -> v1 과 점수가 다른 문항 5개 (open 4 / closed 1)
     0.00->1.00 gt='x-ray'                      pred='chest xray'
     0.00->0.33 gt='Plain film x-ray'           pred='chest xray'
     0.33->0.50 gt='3.4 cm'                     pred='34 cm'
     0.33->0.50 gt='3.4 cm'                     pred='34 cm'
     0.00->1.00 gt='Right side'                 pred='right side'
clean (val 선택 epoch 28)            v1     62.98   83.46    75.33
clean (val 선택 epoch 28)            v2     63.91   83.82    75.92
  -> v1 과 점수가 다른 문항 5개 (open 4 / closed 1)
     0.00->1.00 gt='x-ray'                      pred='chest xray'
     0.00->0.33 gt='Plain film x-ray'           pred='chest xray'
     0.33->0.50 gt='3.4 cm'                     pred='34 cm'
     0.33->0.50 gt='3.4 cm'                     pred='34 cm'
     0.00->1.00

## 8. 학습 곡선

선택은 val 로만 합니다. test 는 평가된 epoch 만 진단용으로 겹쳐 그립니다. `curve_plot.py` 를 먼저 저장하는 셀을 실행하세요.

In [ ]:
%%writefile /content/curve_plot.py
"""모델 공통 학습 곡선 그림 (2x2). 선택 기준은 항상 val 이고, test 곡선은 진단용으로만 겹쳐 그린다.

  (a) train loss (선택: 학습률)
  (b) val overall vs test overall, 세로선 = val 로 고른 epoch
  (c) epoch 별 open / closed / overall  (실선 = val, 점선 = test, 회색 가로 점선 = 논문 test 값)
  (d) "이 epoch 에서 학습을 멈췄다면" 보고했을 값: 그 시점까지 val overall 이 가장 높았던 epoch 의 test 값 (test 가 있는 epoch 만).
      test 가 없으면 val 의 best-so-far 를 그린다. 어느 쪽이든 선택에 test 는 쓰지 않는다.

입력 형식 (epoch 는 0 부터 센 정수 또는 1 부터 센 정수 중 하나로 통일해서 넣을 것)
  train_loss : {epoch: float}
  lr         : {epoch: float}                (선택)
  val, test  : {epoch: {'open': %, 'closed': %, 'overall': %}}   test 는 평가한 epoch 만 있어도 된다
  paper      : {'open': %, 'closed': %, 'overall': %}            (선택)
"""
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

KEYS = ('open', 'closed', 'overall')
COL = {'open': 'tab:blue', 'closed': 'tab:orange', 'overall': 'tab:green'}


def select_by_val(val):
    """val overall 이 최대인 epoch (동점이면 이른 epoch)."""
    return max(sorted(val), key=lambda e: (val[e]['overall'], -e))


def best_so_far(val):
    """각 epoch 까지의 val-best epoch 를 {epoch: 그때의 best epoch} 로 돌려준다."""
    out, best = {}, None
    for e in sorted(val):
        if best is None or val[e]['overall'] > val[best]['overall']:
            best = e
        out[e] = best
    return out


def _sparse(test, val):
    """test 가 epoch 의 절반 미만이면 선으로 잇지 않고 점으로만 그린다 (듬성듬성한 점을 잇는 선은 곡선처럼 오해된다)."""
    return len(test) < max(3, 0.5 * len(val))


def _bars(a, groups, series, ylim=None):
    """groups: 막대 묶음 이름 목록, series: [(label, [값...], 스타일 dict)]"""
    n = len(series); w = 0.8 / n
    for j, (lab, vals, kw) in enumerate(series):
        xs = [i + (j - (n - 1) / 2) * w for i in range(len(groups))]
        a.bar(xs, [0 if v is None else v for v in vals], w * 0.95, label=lab, **kw)
        for x, v in zip(xs, vals):
            if v is not None:
                a.text(x, v + 0.8, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
    a.set_xticks(range(len(groups))); a.set_xticklabels(groups); a.set_ylabel('%')
    if ylim:
        a.set_ylim(*ylim)


def _ticks(axes, val, test, train_loss):
    last = max([max(val)] + ([max(test)] if test else []) + ([max(train_loss)] if train_loss else []))
    step = 5 if last >= 15 else 1
    for a in axes:
        a.xaxis.set_major_locator(MultipleLocator(step)); a.xaxis.set_major_formatter(FormatStrFormatter('%d')); a.grid(axis='x', alpha=0.2)


def _plot_report(title, train_loss, lr, val, test, paper, coverage, out_png, sel):
    """test 를 한두 번만 평가한 경우의 레이아웃: 선택 근거(val) + 최종 test 비교 + 후보 coverage."""
    fig, ax = plt.subplots(2, 2, figsize=(14, 8.5))
    # (a) train loss
    a = ax[0, 0]
    if train_loss:
        xs = sorted(train_loss); a.plot(xs, [train_loss[e] for e in xs], marker='.', label='train loss'); a.set_yscale('log'); a.legend(loc='upper right')
    else:
        a.text(0.5, 0.5, 'no train-loss log found', ha='center', va='center', transform=a.transAxes, color='gray')
    a.set_title('(a) train loss'); a.set_xlabel('epoch')
    # (b) val 곡선 = 선택 근거
    a = ax[0, 1]
    for k in KEYS:
        xs = sorted(val); a.plot(xs, [val[e][k] for e in xs], color=COL[k], lw=1.6 if k == 'overall' else 1.0, label=f'val {k}')
    a.axvline(sel, color='green', lw=1.0, label=f'val-selected epoch {sel}')
    a.set_title('(b) validation per epoch (the only signal used for selection)', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right', fontsize=8)
    # (c) test 비교
    a = ax[1, 0]
    series = []
    if sel in test:
        series.append((f'val-selected (epoch {sel})', [test[sel][k] for k in KEYS], dict(color='tab:blue')))
    last = max(test) if test else None
    if last is not None and last != sel:
        series.append((f'last epoch ({last})', [test[last][k] for k in KEYS], dict(color='tab:cyan')))
    if paper:
        series.append(('paper', [paper.get(k) for k in KEYS], dict(color='lightgray', hatch='//', edgecolor='gray')))
    if series:
        _bars(a, KEYS, series, ylim=(0, 120)); a.legend(loc='upper center', fontsize=8, ncol=3)
    a.set_title('(c) test score, evaluated once per checkpoint (never used for selection)', fontsize=10)
    # (d) candidate coverage
    a = ax[1, 1]
    if coverage:
        cov = [coverage[k]['coverage'] if isinstance(coverage[k], dict) else coverage[k] for k in KEYS]
        _bars(a, KEYS, [('candidate coverage (answer is in the train-only list)', cov, dict(color='lightsteelblue'))], ylim=(0, 120))
        if sel in test:
            a.plot(range(len(KEYS)), [test[sel][k] for k in KEYS], ls='none', marker='D', ms=8, color='tab:blue', markeredgecolor='k', label=f'test score (epoch {sel})')
        a.set_title('(d) candidate coverage of test answers vs achieved score', fontsize=10); a.legend(loc='upper center', fontsize=8, ncol=2)
    else:
        bs = best_so_far(val); xs = sorted(bs)
        for k in KEYS:
            a.step(xs, [val[bs[e]][k] for e in xs], where='post', color=COL[k], label=f'val {k}')
        a.axvline(sel, color='green', lw=1.0)
        a.set_title('(d) val best-so-far (selection trace)', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right', fontsize=8)
    if sel in test:
        t = test[sel]; head = f"val-selected epoch {sel}: test overall {t['overall']:.2f} / open {t['open']:.2f} / closed {t['closed']:.2f}"
    else:
        v = val[sel]; head = f"val-selected epoch {sel}: val overall {v['overall']:.2f} / open {v['open']:.2f} / closed {v['closed']:.2f} (test not evaluated)"
    fig.suptitle(f'{title}  |  {head}', fontsize=11)
    _ticks([ax[0, 0], ax[0, 1]] + ([] if coverage else [ax[1, 1]]), val, test, train_loss)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, sel


def closed_confusion(y_true, y_pred):
    """yes/no 문항의 혼동행렬과 클래스별 precision / recall / F1. y_pred 가 yes/no 가 아니면 'other' 로 센다.
    입력은 소문자·구두점 정리된 문자열 목록. 점수(확률)가 아니라 예측 답만 쓴다."""
    cls = ('yes', 'no'); cols = ('yes', 'no', 'other')
    cm = {t: {c: 0 for c in cols} for t in cls}
    for t, p in zip(y_true, y_pred):
        if t in cm:
            cm[t][p if p in ('yes', 'no') else 'other'] += 1
    m = {}
    for c in cls:
        tp = cm[c][c]; pred = sum(cm[t][c] for t in cls); true = sum(cm[c].values())
        pr = tp / pred if pred else 0.0; rc = tp / true if true else 0.0
        m[c] = {'precision': 100 * pr, 'recall': 100 * rc, 'f1': 100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 'support': true}
    m['macro'] = {k: sum(m[c][k] for c in cls) / 2 for k in ('precision', 'recall', 'f1')}
    n = sum(sum(r.values()) for r in cm.values())
    m['accuracy'] = 100 * sum(cm[c][c] for c in cls) / n if n else 0.0
    return cm, m


def plot_closed_f1(title, y_true, y_pred, out_png=None):
    """closed(yes/no) 예측의 혼동행렬 + 클래스별 precision/recall/F1 막대. (ROC 는 확률 점수가 있어야 해서 여기서는 그리지 않는다)"""
    cm, m = closed_confusion(y_true, y_pred)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    cols = ('yes', 'no', 'other'); rows = ('yes', 'no')
    a = ax[0]; a.imshow([[cm[r][c] for c in cols] for r in rows], cmap='Blues')
    for i, r in enumerate(rows):
        for j, c in enumerate(cols):
            a.text(j, i, cm[r][c], ha='center', va='center', fontsize=14, color='black')
    a.set_xticks(range(3)); a.set_xticklabels([f'pred {c}' for c in cols]); a.set_yticks(range(2)); a.set_yticklabels([f'true {r}' for r in rows])
    a.set_title(f'confusion matrix (closed, n={sum(sum(v.values()) for v in cm.values())})')
    groups = ('yes', 'no', 'macro')
    _bars(ax[1], groups, [(k, [m[g][k] for g in groups], dict(color=c)) for k, c in (('precision', 'tab:blue'), ('recall', 'tab:orange'), ('f1', 'tab:green'))], ylim=(0, 120))
    ax[1].legend(loc='upper center', fontsize=8, ncol=3); ax[1].set_title(f"precision / recall / F1  (accuracy {m['accuracy']:.1f}%)")
    fig.suptitle(title, fontsize=11); fig.tight_layout(rect=(0, 0, 1, 0.94))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, m


def plot_curves(title, train_loss=None, lr=None, val=None, test=None, paper=None, out_png=None, selected=None, coverage=None):
    val, test = val or {}, test or {}
    sparse = _sparse(test, val)
    if sparse:      # test 가 한두 번만 평가된 경우: 우리 방식(val 선택, test 한 번)에 맞는 레이아웃
        assert val, 'val 곡선이 없습니다 (선택 기준이 필요)'
        return _plot_report(title, train_loss, lr, val, test, paper, coverage, out_png, select_by_val(val) if selected is None else selected)
    tkw = dict(ls='none', marker='o', ms=7) if sparse else dict(marker='o', ms=4)
    assert val, 'val 곡선이 없습니다 (선택 기준이 필요)'
    sel = select_by_val(val) if selected is None else selected
    fig, ax = plt.subplots(2, 2, figsize=(14, 8.5))

    # (a) loss
    a = ax[0, 0]
    if train_loss:
        xs = sorted(train_loss)
        a.plot(xs, [train_loss[e] for e in xs], marker='.', label='train loss')
        a.set_yscale('log')
    if not train_loss:
        a.text(0.5, 0.5, 'no train-loss log found', ha='center', va='center', transform=a.transAxes, color='gray')
    a.set_title('(a) train loss' + (' / learning rate' if lr else '')); a.set_xlabel('epoch')
    h, l = a.get_legend_handles_labels()
    if lr:
        b = a.twinx(); xs = sorted(lr)
        b.plot(xs, [lr[e] for e in xs], ':', color='gray', label='lr'); b.set_ylabel('lr')
        h2, l2 = b.get_legend_handles_labels(); h += h2; l += l2
    if h:
        a.legend(h, l, loc='upper right')

    # (b) val vs test overall
    a = ax[0, 1]
    xs = sorted(val); a.plot(xs, [val[e]['overall'] for e in xs], marker='.', color='tab:orange', label='validation overall')
    if test:
        xt = sorted(test); a.plot(xt, [test[e]['overall'] for e in xt], color='tab:blue', label='test overall (diagnostic)', **tkw)
    a.axvline(sel, color='green', lw=0.8, label=f'val-selected epoch {sel}')
    a.set_title('(b) validation vs test overall'); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right')

    # (c) per-epoch metrics
    a = ax[1, 0]
    for k in KEYS:
        xs = sorted(val); a.plot(xs, [val[e][k] for e in xs], color=COL[k], lw=1.2, label=f'val {k}')
        if test:
            xt = sorted(test); a.plot(xt, [test[e][k] for e in xt], color=COL[k], label=f'test {k}', **({'ls': 'none', 'marker': 'o', 'ms': 7} if sparse else {'ls': '--', 'marker': 'o', 'ms': 3, 'lw': 0.9}))
        if paper and paper.get(k) is not None:
            a.axhline(paper[k], color=COL[k], ls=':', lw=0.9, alpha=0.7)
    a.set_title('(c) accuracy per epoch (solid = val, dashed = test, dotted = paper test)'); a.set_xlabel('epoch'); a.set_ylabel('%')
    a.legend(loc='lower right', fontsize=7, ncol=2)

    # (d) reported value if training stopped at this epoch (val-based selection)
    a = ax[1, 1]; bs = best_so_far(val)
    if test and not sparse and any(bs[e] in test for e in bs):
        lab = 'test @ val-best-so-far epoch'
        for k in KEYS:
            pts = [(e, test[bs[e]][k]) for e in sorted(bs) if bs[e] in test]
            a.step([p[0] for p in pts], [p[1] for p in pts], where='post', color=COL[k], label=f'{k} (test)')
    else:
        lab = 'lines: val best-so-far | markers: test at evaluated epochs' if test else 'val best-so-far (no per-epoch test)'
        for k in KEYS:
            a.step(sorted(bs), [val[bs[e]][k] for e in sorted(bs)], where='post', color=COL[k], label=f'{k} (val)')
            if test:
                a.plot(sorted(test), [test[e][k] for e in sorted(test)], ls='none', marker='*', ms=11, color=COL[k], markeredgecolor='k', label=f'{k} (test)')
    if paper:
        for k in KEYS:
            if paper.get(k) is not None:
                a.axhline(paper[k], color='gray', ls=':', lw=0.7)
    a.axvline(sel, color='red', lw=0.8)
    a.set_title('(d) reported value if training stopped at this epoch\n[' + lab + ']', fontsize=10); a.set_xlabel('epoch'); a.set_ylabel('%'); a.legend(loc='lower right')

    if sel in test:
        t = test[sel]; head = f"val-selected epoch {sel}: test overall {t['overall']:.2f} / open {t['open']:.2f} / closed {t['closed']:.2f}"
    else:
        v = val[sel]; head = f"val-selected epoch {sel}: val overall {v['overall']:.2f} / open {v['open']:.2f} / closed {v['closed']:.2f} (test not evaluated)"
    fig.suptitle(f'{title}  |  {head}', fontsize=11)
    # x 축 눈금: epoch 가 15 이상이면 5 단위, 그보다 짧으면 1 단위
    last = max([max(val)] + ([max(test)] if test else []) + ([max(train_loss)] if train_loss else []))
    step = 5 if last >= 15 else 1
    for a in ax.ravel():
        a.xaxis.set_major_locator(MultipleLocator(step)); a.xaxis.set_major_formatter(FormatStrFormatter('%d'))
        a.grid(axis='x', alpha=0.2)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    if out_png:
        fig.savefig(out_png, dpi=130)
    return fig, sel


Overwriting /content/curve_plot.py


In [ ]:
# 8. 학습 곡선 — 선택은 val 로만 한다. test 는 평가된 epoch 만 진단용으로 겹쳐 그린다 (전 epoch test 곡선이 필요하면 EVAL_EVERY=1 로 '_curve' 재실행).
import sys, json, os
sys.path.insert(0, '/content')
import importlib, curve_plot; importlib.reload(curve_plot)      # 저장 셀로 갱신한 최신 코드를 쓰도록 다시 불러온다
from curve_plot import plot_curves, select_by_val as _plot_sel

# 학습 셀 출력(Averaged stats ... loss)에서 옮긴 VQA-RAD(rad_common)·SLAKE(slake_val_sel) 40 epoch 별 train loss — 로그 파일에서 못 읽을 때만 쓴다
LOSS_MANUAL = {**{'VQA-RAD': {0: 12.042789, 1: 7.934632, 2: 5.549639, 3: 4.119543, 4: 3.026361, 5: 2.277315, 6: 1.717127, 7: 1.282052, 8: 1.004648, 9: 0.8575, 10: 0.613725, 11: 0.519847, 12: 0.413643, 13: 0.348782, 14: 0.31707, 15: 0.317863, 16: 0.260279, 17: 0.211901, 18: 0.199439, 19: 0.176169, 20: 0.172454, 21: 0.166265, 22: 0.136475, 23: 0.130949, 24: 0.125998, 25: 0.106522, 26: 0.09841, 27: 0.103754, 28: 0.089958, 29: 0.086491, 30: 0.079054, 31: 0.078111, 32: 0.077982, 33: 0.070564, 34: 0.0751, 35: 0.073394, 36: 0.072833, 37: 0.066792, 38: 0.067204, 39: 0.066396}, 'SLAKE': {0: 6.290528, 1: 2.711381, 2: 1.674811, 3: 1.179942, 4: 0.856733, 5: 0.691055, 6: 0.610606, 7: 0.495935, 8: 0.431192, 9: 0.386581, 10: 0.375481, 11: 0.289885, 12: 0.243431, 13: 0.221805, 14: 0.204643, 15: 0.171819, 16: 0.15, 17: 0.157209, 18: 0.117542, 19: 0.101896, 20: 0.088665, 21: 0.084073, 22: 0.080701, 23: 0.066066, 24: 0.066685, 25: 0.055656, 26: 0.051657, 27: 0.044609, 28: 0.046551, 29: 0.040232, 30: 0.036197, 31: 0.039035, 32: 0.031855, 33: 0.032412, 34: 0.027383, 35: 0.0248, 36: 0.027561, 37: 0.027635, 38: 0.03012, 39: 0.026223}}, **globals().get('LOSS_MANUAL', {})}      # 8-0 셀이 읽은 값이 우선

RUNS = [('VQA-RAD', f'{DRIVE_ROOT}/output/{RAD_OUT_NAME}', f'{DATA_RAD_DIR}/valset.json', f'{DATA_RAD_DIR}/testset.json'),
        ('SLAKE',   f'{DRIVE_ROOT}/output/slake_val_sel', f'{DATA_SLAKE_DIR}/en/slake_val.json', f'{DATA_SLAKE_DIR}/en/slake_test.json')]

def mumc_curves(ds, out_dir, val_file, test_file, title):
    if not _epochs(out_dir, 'val'):
        print(f'  [{ds}] val 결과가 없어(val 선택 없는 프로토콜) 곡선을 생략합니다'); return None
    vals, tests = _epochs(out_dir, 'val'), _epochs(out_dir, 'vqa')
    pick = lambda m: {k: m[k] for k in ('open', 'closed', 'overall')}
    val  = {e: pick(score_result(val_file, f)) for e, f in vals.items()}
    test = {e: pick(score_result(test_file, f)) for e, f in tests.items()}
    loss = {}
    # 1) 레포가 쓰는 log.txt (json 한 줄 = 한 epoch), 2) 학습 셀 출력을 tee 로 저장한 train_stdout.log 의 'Averaged stats' 줄
    lg = os.path.join(out_dir, 'log.txt')
    if os.path.exists(lg):
        for line in open(lg):
            try:
                d = json.loads(line); loss[int(d['epoch'])] = float(d['train_loss'])
            except (ValueError, KeyError, TypeError):
                pass
    so = os.path.join(out_dir, 'train_stdout.log')
    if not loss and os.path.exists(so):
        loss = {int(e): float(l) for e, l in re.findall(r'Train Epoch: \[(\d+)\] Total time.*?\nAveraged stats: lr: [\d.]+\s+loss: ([\d.]+)', open(so).read())}
    if not loss and ds in LOSS_MANUAL and len(LOSS_MANUAL[ds]) == len(val):
        loss = dict(LOSS_MANUAL[ds]); print(f'  ※ {ds}: 로그 파일이 없어 학습 셀 출력에서 옮겨 둔 loss 값을 사용합니다')
    if not loss:
        print(f'  ※ {ds}: train loss 를 찾지 못했습니다. {out_dir} 내용: {sorted(os.listdir(out_dir))[:20]}')
    sel, _ = select_by_val(out_dir, val_file)            # 학습·결과표와 같은 선택 기준 (val pre_answer 완전일치)
    if _plot_sel(val) != sel:
        print(f'  ※ val overall(원 채점) 최고 epoch {_plot_sel(val)} 와 학습 선택 epoch {sel} 가 다릅니다 — 보고값은 {sel} 기준')
    out_png = f'{out_dir}/curves.png'
    fig, _ = plot_curves(title, train_loss=loss, val=val, test=test, paper=paper.get(ds), out_png=out_png, selected=sel,
                         coverage=(cov['test'] if ds == 'VQA-RAD' and 'cov' in globals() else None))   # cov: 결과 셀의 candidate coverage
    print(f'  저장: {out_png}  (val epoch {len(val)}개, test epoch {sorted(test)})')
    return fig

for _ds, _out, _vf, _tf in RUNS:
    try:
        mumc_curves(_ds, _out, _vf, _tf, f'MUMC {_ds} ({RAD_SPLIT if _ds == "VQA-RAD" else "official"} split)')
    except (AssertionError, OSError) as e:
        print(f'[{_ds}] 곡선 생략:', e)


In [ ]:
# 9. closed(yes/no) 혼동행렬 + precision / recall / F1 — val 로 고른 epoch 의 test 예측만 사용한다.
# (ROC 는 yes 확률 같은 점수가 있어야 그릴 수 있는데, 결과 파일에는 최종 예측 답만 저장돼 있어 그리지 않는다)
import importlib, curve_plot; importlib.reload(curve_plot)
from curve_plot import plot_closed_f1

def _cls(text):
    tk = normalize(text); y, n = 'yes' in tk, 'no' in tk
    return 'yes' if y and not n else 'no' if n and not y else 'other'

for _ds, _out, _vf, _tf in RUNS:
    try:
        _sel = select_by_val(_out, _vf)[0] if _epochs(_out, 'val') else max(_epochs(_out, 'vqa'))      # val 선택이 없는 프로토콜은 마지막 epoch
        _pred = {int(r['qid']): r['answer'] for r in json.load(open(_epochs(_out, 'vqa')[_sel]))}
        _rows = [r for r in json.load(open(_tf)) if r['answer_type'] == 'CLOSED' and str(r['answer']).strip().lower() in ('yes', 'no')]
        _fig, _m = plot_closed_f1(f'MUMC {_ds} closed yes/no (n={len(_rows)}) - val-selected epoch {_sel}',
                                  [str(r['answer']).strip().lower() for r in _rows], [_cls(_pred[int(r['qid'])]) for r in _rows], out_png=f'{_out}/closed_f1.png')
        print(f"[{_ds}] accuracy {_m['accuracy']:.2f} | macro F1 {_m['macro']['f1']:.2f} | yes F1 {_m['yes']['f1']:.2f} (n={_m['yes']['support']}) | no F1 {_m['no']['f1']:.2f} (n={_m['no']['support']})")
    except (AssertionError, OSError, KeyError) as e:
        print(f'[{_ds}] closed F1 생략:', repr(e))


In [5]:
import os, glob, sys, subprocess
if not os.path.isdir('/content/Expert-CFG'):
    subprocess.run(['git','clone','-b','VQA_RAD_reproduction','https://github.com/MinseongKim-dev/Expert-CFG','/content/Expert-CFG'], check=True)
else:
    subprocess.run(['git','-C','/content/Expert-CFG','pull'], check=True)
sys.path.insert(0, '/content/Expert-CFG')
print('vqa_scoring.py 있음:', os.path.exists('/content/Expert-CFG/vqa_scoring.py'))

import vqa_scoring as vs
R = '/content/drive/MyDrive/expert_cfg_mumc/output/rad_common/result'
print(sorted(os.path.basename(f) for f in glob.glob(f'{R}/*_vqa_result_*.json')))
for ep in (28, 29, 39):
    f = f'{R}/med_pretrain_29_vqa_result_{ep}.json'
    if os.path.exists(f):
        rec = vs.load_mumc('/content/drive/MyDrive/vqarad_common/testset.json', f)
        print(ep, vs.summarize_unified(rec) if hasattr(vs, 'summarize_unified') else vs.summarize(rec, vs.score_phi))
    else:
        print(ep, '파일 없음')


vqa_scoring.py 있음: True
['med_pretrain_29_vqa_result_28.json', 'med_pretrain_29_vqa_result_28_acc.json', 'med_pretrain_29_vqa_result_28_compare.json', 'med_pretrain_29_vqa_result_39.json', 'med_pretrain_29_vqa_result_39_acc.json', 'med_pretrain_29_vqa_result_39_compare.json']
28 {'closed_acc': 83.08823529411765, 'open_em': 58.659217877094974, 'open_f1': 63.04158907510863, 'overall_acc': 73.39246119733924, 'open_recall(ref)': 63.1657355679702, 'n_closed': 272, 'n_open': 179}
29 파일 없음
39 {'closed_acc': 82.72058823529412, 'open_em': 59.77653631284916, 'open_f1': 64.51272501551831, 'overall_acc': 73.61419068736141, 'open_recall(ref)': 64.00372439478585, 'n_closed': 272, 'n_open': 179}


In [12]:
import os, sys, subprocess, json
if not os.path.isdir('/content/Expert-CFG'):
    subprocess.run(['git','clone','-b','VQA_RAD_reproduction','https://github.com/MinseongKim-dev/Expert-CFG','/content/Expert-CFG'], check=True)
else:
    subprocess.run(['git','-C','/content/Expert-CFG','pull'], check=True)
sys.path.insert(0, '/content/Expert-CFG')
import vqa_scoring as vs

D  = '/content/drive/MyDrive'
EV = '/content/llava_eval_data'          # Colab 로컬: 없으면 LLaVA 노트북의 평가 데이터 셀을 먼저 실행
PAPER = {
  'MUMC VQA-RAD':   (71.50, 84.20, 79.20),
  'LLaVA VQA-RAD':  (61.52, 84.19, 75.19),
  'LLaVA SLAKE':    (83.08, 85.34, 83.97),
}
runs = {}
R = f'{D}/expert_cfg_mumc/output/rad_common/result'
for ep, tag in ((28, 'val 선택 ep28'), (39, '마지막 ep39')):
    runs[f'MUMC VQA-RAD / {tag}'] = ('MUMC VQA-RAD', vs.load_mumc(f'{D}/vqarad_common/testset.json', f'{R}/med_pretrain_29_vqa_result_{ep}.json'))
for name, ref, pred in (('LLaVA VQA-RAD', f'{EV}/vqarad_ref.json', f'{D}/expert_cfg_llavamed/vqarad_pred_greedy.jsonl'),
                        ('LLaVA SLAKE',   f'{EV}/slake_ref.json',  f'{D}/expert_cfg_llavamed/slake_qlora_best_pred.jsonl')):
    if os.path.exists(ref) and os.path.exists(pred):
        runs[f'{name} / val 선택'] = (name, vs.load_llava(ref, pred))
    else:
        print('건너뜀(파일 없음):', name, '| ref:', os.path.exists(ref), '| pred:', os.path.exists(pred))

print(f"{'행':32s}{'채점':>9s}{'open':>8s}{'closed':>8s}{'ALL':>8s}{'논문ALL':>9s}{'격차':>8s}")
for k, (pk, rec) in runs.items():
    p = PAPER[pk]
    for lab, sc in (('원본', vs.score_phi), ('S1+S2', vs.score_s1s2)):
        m = vs.summarize(rec, sc)
        print(f"{k:32s}{lab:>9s}{m['open']:8.2f}{m['closed']:8.2f}{m['overall']:8.2f}{p[2]:9.2f}{m['overall']-p[2]:+8.2f}")


건너뜀(파일 없음): LLaVA VQA-RAD | ref: False | pred: True
건너뜀(파일 없음): LLaVA SLAKE | ref: False | pred: True
행                                      채점    open  closed     ALL    논문ALL      격차
MUMC VQA-RAD / val 선택 ep28             원본   62.98   83.46   75.33    79.20   -3.87
MUMC VQA-RAD / val 선택 ep28          S1+S2   65.12   83.46   76.18    79.20   -3.02
MUMC VQA-RAD / 마지막 ep39                원본   64.00   82.72   75.29    79.20   -3.91
MUMC VQA-RAD / 마지막 ep39             S1+S2   66.61   82.72   76.33    79.20   -2.87
